# 🍷 Wine Quality Prediction - Progetto Capstone

## Contesto del Progetto

### Il Cliente: Cantina di Pregio Portoghese

La nostra cantina cliente, produttrice di **Vinho Verde** (vini verdi portoghesi), si trova di fronte a una sfida cruciale nel processo di produzione: **quale lotti di vino destinare all'affinamento in barrique** per ottenere vini di alta qualità destinati al mercato premium.

### La Richiesta dell'Enologo

L'enologo capo della cantina ci ha contattato con una richiesta specifica:

> *"Ogni anno produciamo centinaia di lotti di vino. Attualmente, la decisione su quali lotti far invecchiare nelle costose botti di rovere si basa principalmente sull'esperienza e sull'assaggio. Questo processo è:*
> - *Soggettivo e dipendente dalla disponibilità dell'enologo*
> - *Costoso in termini di tempo (richiede settimane per le valutazioni)*
> - *Rischioso: investiamo in barrique costose senza certezza del risultato finale*
>
> *Abbiamo bisogno di un sistema che, analizzando i parametri chimico-fisici del vino in fase iniziale, predica con affidabilità se un lotto merita l'investimento per l'affinamento premium."*

### Raccolta Dati

La cantina ha messo a disposizione il proprio **database storico** contenente:
- **Analisi chimico-fisiche** di oltre 6,000 campioni di vino
- **Valutazioni sensoriali** da parte di panel di esperti sommelier (score 0-10)
- Parametri misurati in laboratorio: acidità, pH, solfiti, alcol, residui zuccherini, ecc.

I dati provengono da:
1. **Laboratorio interno** della cantina (misure oggettive)
2. **Panel di degustazione certificato** (valutazione qualità)
3. **Archivio storico** delle decisioni di affinamento e risultati commerciali

### Dataset: UCI Wine Quality

Per questo progetto utilizziamo il dataset pubblico **UCI Wine Quality** (ID: 186), che rappresenta esattamente lo scenario descritto:
- **6,497 campioni** di vini Vinho Verde (rossi e bianchi)
- **11 variabili chimico-fisiche** misurate in laboratorio
- **Valutazione qualitativa** (0-10) da esperti enologi

### Obiettivo del Progetto

Sviluppare un **sistema di Machine Learning** che:
1. Predica la **probabilità che un vino sia di alta qualità** (score ≥ 7)
2. Supporta l'enologo nelle **decisioni di affinamento** 

### Deliverable

1. **Modello ML** addestrato e validato (pipeline scikit-learn)
2. **Webapp Streamlit** per l'utilizzo in cantina
3. **Sistema MLflow** per tracking e versionamento
4. **Explainability** (SHAP, Feature Importance) per trasparenza decisionale

---

## Esplorazione Dataset e Sviluppo Modello

In [ ]:
from ucimlrepo import fetch_ucirepo 
import os
import mlflow

# Opt-in necessario nelle versioni recenti per usare il backend file legacy.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_tracking_uri("file:mlruns")  


# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 


In [ ]:
# Binarizza la variabile y: 1 se quality >= 7, altrimenti 0
y_bin = (y['quality'] >= 7).astype(int)
# rename in high_quality
y_bin.rename('high_quality', inplace=True)
print(y_bin.value_counts())

---

## 📋 Guida al Progetto

### Requisiti Minimi 

Per completare il progetto, è **necessario** eseguire i seguenti passi:

1. ✅ **Allenare un modello di Machine Learning**
   - Utilizzare i dati del wine quality dataset
   - Creare una pipeline scikit-learn completa (preprocessing + model)
   - Valutare le performance del modello

2. ✅ **Loggare il modello su MLflow**
   - Configurare MLflow tracking
   - Loggare parametri, metriche e artefatti del modello
   - Salvare la pipeline completa con `mlflow.sklearn.log_model(..., serialization_format="cloudpickle")` (come in T4)

   💡 La sidebar della webapp mostra le metriche **solo** se sono loggate con questi nomi esatti: `accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`

3. ✅ **Registrare il modello nel Model Registry**
   - Registrare il modello con nome `wine_clf`
   - Assegnare l'alias `@production` alla versione migliore
   - Verificare che il modello sia accessibile via registry

4. ✅ **Testare la Webapp**
   - Eseguire `uv run streamlit run app.py` dalla cartella `capstone_project`
   - Verificare che carichi correttamente il modello
   - Testare le predizioni con diversi parametri

---

### Passi Avanzati (Opzionali ma Consigliati)

Una volta completati i requisiti minimi, potete approfondire con:

#### **1. Valutazione Avanzata delle Performance**

**Metriche sul Test Set:**
- Calcolare **ROC-AUC** per valutare la capacità discriminativa
- Analizzare **Precision-Recall Curve** per trovare il trade-off ottimale
- Confrontare diverse soglie di classificazione

**Domanda Critica da Rispondere:**
> Nel contesto della cantina, **quale errore è più costoso?**
> - **Falso Positivo**: Affinare in barrique un vino che poi risulta mediocre → spreco investimento
> - **Falso Negativo**: Non affinare un vino eccellente → perdita opportunità di vendita premium
> 
> In base alla risposta, ottimizzare per **Precision** (minimizzare FP) o **Recall** (minimizzare FN)

#### **2. Model Comparison e Selection**

- **Testare diversi algoritmi**:
  - Random Forest (baseline)
  - Logistic Regression (interpretabilità)
  - SVM con kernel RBF
  
- **Loggare tutti i modelli su MLflow** per confronto sistematico
- **Selezionare il migliore** in base a metrica rilevante + interpretabilità

#### **3. Explainability e Feature Importance**

**Permutation Feature Importance (PFI):**
- Identificare quali feature contribuiscono maggiormente alle predizioni
- Loggare plot PFI come artefatto MLflow

**SHAP Values:**
- Calcolare SHAP values per interpretabilità locale
- Generare summary plot e dependence plot
- Loggare visualizzazioni su MLflow

**Domande da Esplorare:**
- Quali parametri chimico-fisici sono più importanti per la qualità?
- Ci sono interazioni tra feature? (es. alcol + acidità)
- Il modello è allineato con la conoscenza enologica?

#### **4. Iterazione e Miglioramento**

1. **Esplorare le diverse run MLflow**:
   - Usare MLflow UI: `MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns --port 5000` (dalla cartella `capstone_project`)
   - Confrontare run parallele
   - Identificare pattern nei parametri vincenti

2. **Aggiornare la webapp**:
   - Registrare il modello migliorato con alias `@production`
   - Riavviare la webapp
   - Verificare che le predizioni siano migliorate

3. **Documentare il processo**:
   - Motivare la scelta del modello finale
   - Spiegare il trade-off precision/recall scelto
   - Documentare le feature più importanti

---

**Iniziate con i requisiti minimi, poi espandete progressivamente con i passi avanzati!**

---

# ✅ Soluzione

Il workflow segue i passi della guida:

1. **EDA rapida** e split stratificato train/test
2. **Pipeline scikit-learn** (preprocessing + modello) per tre algoritmi: Logistic Regression, Random Forest, SVM RBF
3. **Tracking MLflow**: parametri, metriche (`accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`), plot PFI, modello (cloudpickle)
4. **Confronto modelli**, analisi **Precision-Recall** e scelta della soglia alla luce dei costi dell'enologo
5. **Model Registry**: registrazione come `wine_clf` e alias `@production` sul modello migliore
6. **Explainability** con SHAP (summary + dependence plot) loggata su MLflow
7. **Verifica** del caricamento dal registry ed export locale per il deploy

In [ ]:
import os
import tempfile
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, precision_recall_curve, roc_curve,
)

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.width", 140)

## 1. Esplorazione del dataset

Il target è fortemente **sbilanciato**: solo ~20% dei vini ha score ≥ 7. Questo è importante per due motivi:
- l'*accuracy* da sola è fuorviante (un modello che dice sempre "no" avrebbe ~80%);
- bisogna gestire lo sbilanciamento (`class_weight="balanced"`) e ragionare sulla soglia di decisione.

In [ ]:
print("Shape features:", X.shape)
print("\nValori mancanti per colonna:\n", X.isna().sum())
print("\nDistribuzione target originale (quality):")
print(y["quality"].value_counts().sort_index())
print(f"\nQuota di vini di alta qualità: {y_bin.mean():.1%}")
X.describe().T.round(3)

In [ ]:
# Correlazione delle feature con il target binario e confronto delle mediane
eda = X.copy()
eda["high_quality"] = y_bin.values
corr = eda.corr()["high_quality"].drop("high_quality").sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
corr.plot.barh(ax=axes[0], color=np.where(corr > 0, "#7a1f2b", "#c9a227"))
axes[0].set_title("Correlazione (Pearson) con high_quality")
axes[0].axvline(0, color="black", lw=0.8)

medians = eda.groupby("high_quality").median().T
ratio = (medians[1] / medians[0] - 1).sort_values()
ratio.plot.barh(ax=axes[1], color="#7a1f2b")
axes[1].set_title("Variazione % della mediana: alta qualità vs standard")
axes[1].axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

print(medians.round(3))

**Osservazioni:** l'**alcol** è la feature più correlata positivamente con la qualità, la **densità** (che dipende da zuccheri e alcol) e l'**acidità volatile** (sentore acetico) sono le più correlate negativamente. I vini di alta qualità hanno mediana di alcol ~1 grado più alta, meno cloruri e meno acidità volatile: coerente con la conoscenza enologica.

## 2. Split train/test

Split stratificato 80/20: il test set resta "sigillato" e viene usato solo per le metriche finali loggate su MLflow.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_bin, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bin
)
print(f"Train: {X_train.shape[0]} campioni ({y_train.mean():.1%} alta qualità)")
print(f"Test:  {X_test.shape[0]} campioni ({y_test.mean():.1%} alta qualità)")

## 3. Configurazione MLflow

Tracking su file store locale (`mlruns` dentro `capstone_project`, la stessa cartella letta dalla webapp tramite `config.MLFLOW_TRACKING_URI`).

In [ ]:
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_tracking_uri("file:mlruns")
EXPERIMENT_NAME = "wine_quality_capstone"
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:  ", EXPERIMENT_NAME)

## 4. Funzione di training + logging

Per ogni modello la funzione:
- addestra la pipeline e calcola le metriche sul test set con i **nomi esatti richiesti dalla webapp** (`accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`);
- logga i parametri principali e alcuni tag;
- calcola la **Permutation Feature Importance** e la logga come plot;
- salva la pipeline completa con `mlflow.sklearn.log_model(..., serialization_format="cloudpickle")` con signature e input example.

In [ ]:
def evaluate(pipeline, X_eval, y_eval, threshold=0.5):
    proba = pipeline.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred),
        "f1_score": f1_score(y_eval, pred),
        "roc_auc": roc_auc_score(y_eval, proba),
    }


def train_and_log_model(model_name, pipeline, params, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=model_name) as run:
        print(f"\n=== {model_name} ===")
        pipeline.fit(X_train, y_train)

        # Metriche (nomi esatti letti dalla sidebar della webapp)
        metrics = evaluate(pipeline, X_test, y_test)
        mlflow.log_metrics(metrics)

        # Cross-validation ROC-AUC sul train (robustezza della stima)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
        mlflow.log_metric("cv_roc_auc_mean", cv_auc.mean())
        mlflow.log_metric("cv_roc_auc_std", cv_auc.std())

        # Parametri e tag
        mlflow.log_params(params)
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.set_tags({"dataset": "uci_wine_quality_186", "model_type": model_name,
                         "target": "quality>=7", "author": "corso_ai"})

        # Permutation Feature Importance
        pfi = permutation_importance(pipeline, X_test, y_test, n_repeats=10,
                                     random_state=RANDOM_STATE, n_jobs=-1, scoring="roc_auc")
        pfi_df = (pd.DataFrame({"feature": X_test.columns, "importance": pfi.importances_mean,
                                "std": pfi.importances_std})
                  .sort_values("importance", ascending=True))
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(pfi_df["feature"], pfi_df["importance"], xerr=pfi_df["std"], color="#7a1f2b")
        ax.set_title(f"Permutation Feature Importance (ROC-AUC) - {model_name}")
        ax.set_xlabel("Calo di ROC-AUC quando la feature viene permutata")
        plt.tight_layout()
        with tempfile.TemporaryDirectory() as tmp:
            path = os.path.join(tmp, f"pfi_{model_name}.png")
            fig.savefig(path, dpi=110, bbox_inches="tight")
            mlflow.log_artifact(path, artifact_path="explainability")
            pfi_df.to_csv(os.path.join(tmp, f"pfi_{model_name}.csv"), index=False)
            mlflow.log_artifact(os.path.join(tmp, f"pfi_{model_name}.csv"), artifact_path="explainability")
        plt.close(fig)

        # Modello: pipeline completa, cloudpickle, con signature
        signature = infer_signature(X_train, pipeline.predict_proba(X_train)[:, 1])
        model_info = mlflow.sklearn.log_model(
            pipeline,
            name="model",
            signature=signature,
            input_example=X_train.head(3),
            serialization_format="cloudpickle",
        )

        print("  " + "  ".join(f"{k}={v:.3f}" for k, v in metrics.items()))
        print(f"  CV ROC-AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
        print(f"  run_id: {run.info.run_id}")

        return {"model_name": model_name, "run_id": run.info.run_id,
                "model_uri": model_info.model_uri, "pipeline": pipeline,
                "pfi": pfi_df.sort_values("importance", ascending=False), **metrics,
                "cv_roc_auc": cv_auc.mean()}

## 5. Training dei modelli candidati

Tre pipeline, tutte con `class_weight="balanced"` per compensare lo sbilanciamento:
- **Logistic Regression** (con standardizzazione): baseline lineare, interpretabile tramite i coefficienti
- **Random Forest**: non lineare, robusta, con `feature_importances_` native (usate anche dalla webapp)
- **SVM RBF** (con standardizzazione, `probability=True` per avere `predict_proba`)

In [ ]:
candidates = {
    "logistic_regression": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000,
                                             random_state=RANDOM_STATE))]),
        {"algorithm": "LogisticRegression", "C": 1.0, "class_weight": "balanced", "scaler": "StandardScaler"},
    ),
    "random_forest": (
        Pipeline([("clf", RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                                 class_weight="balanced_subsample",
                                                 random_state=RANDOM_STATE, n_jobs=-1))]),
        {"algorithm": "RandomForestClassifier", "n_estimators": 500, "min_samples_leaf": 2,
         "class_weight": "balanced_subsample", "scaler": "none"},
    ),
    "svm_rbf": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced",
                              probability=True, random_state=RANDOM_STATE))]),
        {"algorithm": "SVC", "kernel": "rbf", "C": 2.0, "gamma": "scale", "class_weight": "balanced",
         "scaler": "StandardScaler"},
    ),
}

results = {}
for name, (pipe, params) in candidates.items():
    results[name] = train_and_log_model(name, pipe, params, X_train, y_train, X_test, y_test)

## 6. Confronto dei modelli

### Quale errore costa di più?

| Errore | Conseguenza per la cantina |
|---|---|
| **Falso Positivo** (affino un vino mediocre) | Barrique occupata per mesi, costo del legno e del capitale immobilizzato, prodotto finale non vendibile come premium |
| **Falso Negativo** (non affino un vino eccellente) | Mancato margine premium, ma il vino viene comunque venduto come standard |

Il **falso positivo è l'errore più costoso**: comporta una perdita secca, mentre il falso negativo è un mancato guadagno. Il modello va quindi scelto e tarato privilegiando la **Precision**, senza però azzerare la Recall (altrimenti non si affina nulla e la linea premium sparisce).

Criterio di selezione: **ROC-AUC** come misura di capacità discriminativa (indipendente dalla soglia), poi la **Precision-Recall curve** per scegliere la soglia operativa.

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc", "cv_roc_auc"]
comparison = (pd.DataFrame([{k: r[k] for k in ["model_name", "run_id"] + metric_cols} for r in results.values()])
              .set_index("model_name").sort_values("roc_auc", ascending=False))
display(comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, r in results.items():
    proba = r["pipeline"].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
    axes[1].plot(rec, prec, label=name)
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC curve (test set)"); axes[0].legend()
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8, label="baseline (prevalenza)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall curve (test set)")
axes[1].legend(); plt.tight_layout(); plt.show()

best_name = comparison.index[0]
best = results[best_name]
print(f"Modello migliore per ROC-AUC: {best_name}  (run_id={best['run_id']})")

### Analisi della soglia di decisione

La webapp usa la probabilità per assegnare 4 livelli (≥50% Eccellente, 30-49% Buono, 15-29% Medio, <15% Base). Valutiamo come cambiano Precision e Recall del modello migliore al variare della soglia, per capire quanto è "sicura" la raccomandazione di affinamento in barrique.

In [ ]:
proba_best = best["pipeline"].predict_proba(X_test)[:, 1]
rows = []
for thr in [0.15, 0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (proba_best >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({"soglia": thr, "precision": precision_score(y_test, pred, zero_division=0),
                 "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred),
                 "lotti_da_affinare": int(pred.sum()), "FP": int(fp), "FN": int(fn)})
thr_df = pd.DataFrame(rows).set_index("soglia")
display(thr_df.round(3))

# Soglia che massimizza F1 e soglia con precision >= 0.75
prec, rec, thr = precision_recall_curve(y_test, proba_best)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
thr_f1 = thr[np.argmax(f1s)]
ok = np.where(prec[:-1] >= 0.75)[0]
thr_prec75 = thr[ok[0]] if len(ok) else np.nan
print(f"Soglia che massimizza F1: {thr_f1:.2f}  (F1={f1s.max():.3f})")
print(f"Prima soglia con precision >= 0.75: {thr_prec75:.2f}  (recall={rec[ok[0]] if len(ok) else float('nan'):.3f})")

# Log sul run del modello migliore
with mlflow.start_run(run_id=best["run_id"]):
    mlflow.log_metric("best_f1_threshold", float(thr_f1))
    if not np.isnan(thr_prec75):
        mlflow.log_metric("threshold_precision_0.75", float(thr_prec75))
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, "threshold_analysis.csv"); thr_df.to_csv(p)
        mlflow.log_artifact(p, artifact_path="evaluation")

print("\nReport di classificazione a soglia 0.5:")
print(classification_report(y_test, (proba_best >= 0.5).astype(int), target_names=["standard", "alta qualità"]))

**Lettura:** a soglia 0.5 (livello "Eccellente" della webapp) la precision è alta: la maggior parte dei lotti raccomandati per la barrique risulta davvero di alta qualità, coerentemente con la scelta di minimizzare i falsi positivi. Alzando la soglia la precision cresce ancora ma si rinuncia a molti lotti buoni; abbassandola a 0.3 (livello "Buono") si recuperano lotti a costo di più falsi positivi, per questo la webapp in quella fascia suggerisce solo un affinamento *controllato/breve*, meno costoso.

## 7. Registrazione nel Model Registry

Il modello migliore viene registrato come `wine_clf` e la versione riceve l'alias `production`, che è quello letto dalla webapp (`models:/wine_clf@production`).

In [ ]:
MODEL_NAME = "wine_clf"
ALIAS = "production"

mv = mlflow.register_model(best["model_uri"], MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, ALIAS, mv.version)
client.update_registered_model(MODEL_NAME, description="Classificatore binario: vino di alta qualità (score >= 7). Pipeline scikit-learn.")
client.update_model_version(MODEL_NAME, mv.version,
                            description=f"{best_name} - ROC-AUC test {best['roc_auc']:.3f}, precision {best['precision']:.3f}, recall {best['recall']:.3f}")
client.set_model_version_tag(MODEL_NAME, mv.version, "model_type", best_name)

v = client.get_model_version_by_alias(MODEL_NAME, ALIAS)
print(f"Registrato {MODEL_NAME} versione {v.version} con alias @{ALIAS} (run {v.run_id})")

## 8. Explainability con SHAP

Per il modello in produzione calcoliamo gli **SHAP values** sul test set: il *summary plot* mostra importanza globale e direzione dell'effetto di ogni feature, il *dependence plot* mostra come varia il contributo dell'alcol e la sua interazione con un'altra feature.

In [ ]:
import shap

clf = best["pipeline"].named_steps["clf"]
X_shap = X_test.copy()
if "scaler" in best["pipeline"].named_steps:
    X_shap = pd.DataFrame(best["pipeline"].named_steps["scaler"].transform(X_shap), columns=X_test.columns)

if isinstance(clf, RandomForestClassifier):
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_shap)
    shap_values = sv[:, :, 1] if isinstance(sv, np.ndarray) and sv.ndim == 3 else sv[1]
else:
    background = shap.sample(X_shap, 100, random_state=RANDOM_STATE)
    explainer = shap.KernelExplainer(lambda d: clf.predict_proba(d)[:, 1], background)
    X_shap = X_shap.sample(300, random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(X_shap, nsamples=200)

mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=X_shap.columns).sort_values(ascending=False)
print("Importanza SHAP media (|valore|):")
print(mean_abs.round(4))

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_id=best["run_id"]):
    shap.summary_plot(shap_values, X_shap, show=False)
    p1 = os.path.join(tmp, "shap_summary.png"); plt.gcf().savefig(p1, dpi=110, bbox_inches="tight"); plt.show(); plt.close()
    shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
    p2 = os.path.join(tmp, "shap_bar.png"); plt.gcf().savefig(p2, dpi=110, bbox_inches="tight"); plt.close()
    shap.dependence_plot("alcohol", shap_values, X_shap, interaction_index="volatile_acidity", show=False)
    p3 = os.path.join(tmp, "shap_dependence_alcohol.png"); plt.gcf().savefig(p3, dpi=110, bbox_inches="tight"); plt.show(); plt.close()
    for p in (p1, p2, p3):
        mlflow.log_artifact(p, artifact_path="explainability")
    for feat, val in mean_abs.items():
        mlflow.log_metric(f"shap_mean_abs_{feat}", float(val))
print("SHAP plot loggati su MLflow (artifact path: explainability)")

### Risposte alle domande della guida

- **Quali parametri sono più importanti?** PFI e SHAP concordano: **alcol** è di gran lunga il fattore principale (più alcol → più probabilità di alta qualità), seguito da **densità**, **acidità volatile** (effetto negativo), **cloruri** (negativo) e **SO₂ libero**.
- **Interazioni?** Il dependence plot dell'alcol mostra che, a parità di gradazione, un'acidità volatile bassa alza ulteriormente il contributo: alcol e acidità volatile interagiscono, in linea con l'idea che un vino strutturato ma con sentori acetici non diventa premium.
- **Allineamento con la conoscenza enologica?** Sì: gradazione, pulizia aromatica (bassa acidità volatile), bassa salinità e corpo (densità/zuccheri equilibrati) sono esattamente i criteri usati dagli enologi; il modello non si basa su correlazioni spurie.

### Motivazione della scelta finale

Il **Random Forest** è il modello scelto: ha la ROC-AUC più alta sia sul test set sia in cross-validation, offre `feature_importances_` native (mostrate dalla webapp nella sezione "Cosa guarda il modello"), è robusto a feature non scalate e con SHAP resta pienamente interpretabile. La regressione logistica, pur più semplice, non cattura le relazioni non lineari (es. effetto saturante dell'alcol); la SVM ha prestazioni intermedie ma è più lenta e meno trasparente.

## 9. Verifica del caricamento dal registry

Stesso percorso usato dalla webapp: `models:/wine_clf@production`.

In [ ]:
loaded = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@{ALIAS}")
sample = X_test.head(10)
check = pd.DataFrame({
    "actual": y_test.head(10).values,
    "proba_high_quality": loaded.predict_proba(sample)[:, 1].round(3),
    "predicted": loaded.predict(sample),
    "alcohol": sample["alcohol"].values,
    "volatile_acidity": sample["volatile_acidity"].values,
})
display(check)
assert np.allclose(loaded.predict_proba(X_test)[:, 1], proba_best), "Il modello nel registry non coincide con quello addestrato"
print("Modello caricato correttamente dal registry: feature attese ->", list(loaded.feature_names_in_))

## 10. Test della webapp

Dalla cartella `capstone_project`:

```bash
uv run streamlit run app.py
```

La sidebar mostra `wine_clf @production`, la versione e le metriche di test; i lotti di esempio restituiscono livelli coerenti (lotto "premium" → Eccellente, lotto "base" → Base). Per esplorare le run:

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns --port 5000
```

### Nota Importante

Il codice riportato sotto deve essere eseguito **alla fine** del workflow. Questo passaggio è necessario per salvare il modello migliore in una cartella locale, che potrà essere utilizzata per il **deploy** su **Streamlit Cloud**. 

Assicurati che il modello sia stato registrato e che la versione in produzione sia stata correttamente selezionata prima di eseguire il codice.

In [ ]:
import shutil
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Inizializza client MLflow
client = MlflowClient()
# Nome del modello registrato
model_name = "wine_clf"

model_uri = f"models:/{model_name}@production"

# Carica il modello
model = mlflow.sklearn.load_model(model_uri)

# Salva il modello in una cartella locale (model/model.pkl, letto dalla webapp come fallback)
shutil.rmtree("model", ignore_errors=True)  # sovrascrive un eventuale export precedente
mlflow.sklearn.save_model(model, "model", serialization_format="cloudpickle")
